# 01 — Clean the rainfall data and complete Task 1

Run this notebook after cloud setup. It reads the eight manifest-listed annual CSVs, applies visible cleaning rules, and describes rainfall patterns across the full 2017–2024 assignment period. The calculations are shown here so the notebook can stand on its own during marking. Task 1 is descriptive; its 2024 summaries are not used for Task 2 model or hyperparameter selection.


In [ ]:
from pathlib import Path
from functools import reduce
import json
import os
import subprocess

from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StringType, StructField, StructType

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"

def read_gcs_json(uri):
    result = subprocess.run(
        ["gcloud", "storage", "cat", uri], check=True, capture_output=True, text=True
    )
    return json.loads(result.stdout)

config = read_gcs_json(f"{ROOT}/config/experiment.json")
assert config["years"] == list(range(2017, 2025))
assert config["alignment_tolerance_seconds"] == 0
spark = SparkSession.builder.appName("dsa5208-rainfall-task1").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(config["shuffle_partitions"]))
spark.conf.set("spark.sql.session.timeZone", config["timezone"])
spark.conf.set("spark.sql.ansi.enabled", "false")
actual_spark_conf = {
    name: spark.conf.get(name)
    for name in ("spark.sql.shuffle.partitions", "spark.sql.session.timeZone", "spark.sql.ansi.enabled")
}
assert actual_spark_conf["spark.sql.shuffle.partitions"] == str(config["shuffle_partitions"])
assert actual_spark_conf["spark.sql.session.timeZone"] == config["timezone"]
assert actual_spark_conf["spark.sql.ansi.enabled"] == "false"
display({"actual_spark_conf": actual_spark_conf})


## Load and inspect the eight annual files

The manifest selects the input files. Each annual DataFrame gets its source year before the rows are combined, so row counts and exclusions can be traced back to the file.

In [ ]:
manifest = read_gcs_json(f"{ROOT}/config/input_manifest.json")
files = sorted(manifest["files"], key=lambda row: row["year"])
assert [row["year"] for row in files] == list(range(2017, 2025))

column_names = [
    "date", "timestamp", "update_timestamp", "station_id", "station_name",
    "station_device_id", "location_longitude", "location_latitude",
    "reading_update_timestamp", "reading_value", "reading_type", "reading_unit",
]
raw_schema = StructType(
    [StructField(name, StringType(), True) for name in column_names]
    + [StructField("_corrupt_record", StringType(), True)]
)


In [ ]:
annual_frames = []
for record in files:
    source = f"gs://{BUCKET}/{record['relative_path']}"
    annual = (
        spark.read.schema(raw_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .option("encoding", "UTF-8")
        .csv(source)
        .withColumn("_source_year", F.lit(record["year"]).cast("int"))
        .withColumn("_source_file", F.lit(record["relative_path"]))
    )
    annual_frames.append(annual)
raw = reduce(lambda left, right: left.unionByName(right), annual_frames).persist(StorageLevel.DISK_ONLY)
raw.groupBy("_source_year").count().orderBy("_source_year").show(8, truncate=False)
raw.groupBy("_source_year", "reading_type", "reading_unit").count().orderBy("_source_year", F.desc("count")).show(30, truncate=False)


In [ ]:
def add_parsed_columns(frame):
    parsed = frame
    for name in ("timestamp", "update_timestamp", "reading_update_timestamp"):
        parsed = parsed.withColumn(f"_{name}_parsed", F.to_timestamp(F.trim(F.col(name))))
    parsed = parsed.withColumn("_date_parsed", F.to_date(F.trim(F.col("date"))))
    for name in ("reading_value", "location_longitude", "location_latitude"):
        parsed = parsed.withColumn(f"_{name}_numeric", F.trim(F.col(name)).cast("double"))
    return parsed


def is_finite(column):
    return column.isNotNull() & ~F.isnan(column) & (F.abs(column) != float("inf"))


parsed = add_parsed_columns(raw)
raw_quality = parsed.groupBy("_source_year").agg(
    F.count("*").alias("raw_rows"),
    F.sum(F.col("_corrupt_record").isNotNull().cast("long")).alias("malformed_rows"),
    F.sum(F.col("station_id").isNull().cast("long")).alias("missing_station_rows"),
    F.sum(F.col("_timestamp_parsed").isNull().cast("long")).alias("invalid_timestamp_rows"),
    F.sum(F.col("_reading_value_numeric").isNull().cast("long")).alias("invalid_rainfall_rows"),
)
raw_quality.orderBy("_source_year").show(8, truncate=False)


## Clean, align and deduplicate

Rows are retained only when station, timestamp, rainfall, unit, type and coordinates are valid. Timestamps must already lie on the exact five-minute grid. Conflicting records at one station/time key are excluded; matching repeats are reduced deterministically.

In [ ]:
def flag_reason(frame, condition, reason):
    return frame.withColumn(
        "exclusion_reason",
        F.when(F.col("exclusion_reason").isNotNull(), F.col("exclusion_reason"))
        .when(F.coalesce(condition, F.lit(True)), F.lit(reason)),
    )


def validate_measurement_fields(raw_rows, config):
    frame = add_parsed_columns(raw_rows).withColumn("exclusion_reason", F.lit(None).cast("string"))
    rules = [
        (F.col("_corrupt_record").isNotNull(), "csv_parser_error"),
        (F.col("station_id").isNull() | (F.trim("station_id") == ""), "missing_station"),
        (F.col("_timestamp_parsed").isNull(), "invalid_timestamp"),
        (F.col("_date_parsed").isNull() | (F.col("_date_parsed") != F.to_date("_timestamp_parsed")), "date_mismatch"),
        (F.year("_timestamp_parsed") != F.col("_source_year"), "year_mismatch"),
        (~is_finite(F.col("_reading_value_numeric")), "invalid_rainfall"),
        (F.col("_reading_value_numeric") < 0, "negative_rainfall"),
        (F.trim("reading_unit") != "mm", "invalid_unit"),
        (F.trim("reading_type") != config["reading_type"], "invalid_reading_type"),
        (~is_finite(F.col("_location_longitude_numeric")) | ~is_finite(F.col("_location_latitude_numeric")), "invalid_coordinates"),
        (~F.col("_location_longitude_numeric").between(-180, 180) | ~F.col("_location_latitude_numeric").between(-90, 90), "coordinates_out_of_range"),
    ]
    for condition, reason in rules:
        frame = flag_reason(frame, condition, reason)
    return frame


In [ ]:
def apply_alignment_and_precision_checks(frame, config):
    frame = frame.withColumn("epoch", F.col("_timestamp_parsed").cast("long"))
    frame = frame.withColumn("alignment_offset_seconds", F.pmod("epoch", F.lit(300)))
    alignment = frame.groupBy("_source_year", "alignment_offset_seconds").count()
    frame = flag_reason(frame, F.col("alignment_offset_seconds") != 0, "off_grid_timestamp")
    scale = config["rainfall_integer_scale"]
    frame = frame.withColumn("rain_units", F.round(F.col("_reading_value_numeric") * scale).cast("long"))
    frame = flag_reason(
        frame,
        F.abs(F.col("rain_units") / scale - F.col("_reading_value_numeric")) > 1e-10,
        "unsupported_rainfall_precision",
    )
    return frame, alignment


In [ ]:
def resolve_duplicate_keys(candidates):
    expected = column_names + ["_source_year"]
    exact_order = Window.partitionBy(*expected).orderBy("_source_file")
    ranked = candidates.withColumn("_exact_rank", F.row_number().over(exact_order))
    exact_repeats = ranked.where("_exact_rank > 1").withColumn("exclusion_reason", F.lit("exact_duplicate"))
    unique = ranked.where("_exact_rank = 1")

    variants = unique.groupBy("station_id", "epoch").agg(
        F.countDistinct(F.struct(
            "rain_units", "station_name", "station_device_id",
            "_location_longitude_numeric", "_location_latitude_numeric",
        )).alias("_variants")
    )
    key_order = Window.partitionBy("station_id", "epoch").orderBy(
        "_update_timestamp_parsed", "_reading_update_timestamp_parsed", "_source_file"
    )
    ranked_keys = unique.join(variants, ["station_id", "epoch"]).withColumn(
        "_key_rank", F.row_number().over(key_order)
    )
    conflicts = ranked_keys.where("_variants > 1").withColumn("exclusion_reason", F.lit("conflicting_key"))
    repeats = ranked_keys.where("_variants = 1 AND _key_rank > 1").withColumn(
        "exclusion_reason", F.lit("same_measurement_duplicate")
    )
    retained = ranked_keys.where("_variants = 1 AND _key_rank = 1")
    rejected = exact_repeats.unionByName(conflicts, allowMissingColumns=True).unionByName(
        repeats, allowMissingColumns=True
    )
    return retained, rejected


In [ ]:
def select_clean_columns(retained):
    return retained.select(
        "station_id", "epoch", "rain_units",
        F.col("_timestamp_parsed").alias("observation_time"),
        F.col("_reading_value_numeric").alias("rain_mm"),
        F.col("_location_longitude_numeric").alias("longitude"),
        F.col("_location_latitude_numeric").alias("latitude"),
        F.col("_source_year").alias("year"),
        F.to_date("_timestamp_parsed").alias("local_date"),
        F.month("_timestamp_parsed").alias("month"),
        F.hour("_timestamp_parsed").alias("hour"),
        F.col("_update_timestamp_parsed").alias("update_time"),
        F.col("_reading_update_timestamp_parsed").alias("reading_update_time"),
        "station_name", "station_device_id", "_source_file", "alignment_offset_seconds",
    )


In [ ]:
validated = validate_measurement_fields(raw, config)
validated, alignment = apply_alignment_and_precision_checks(validated, config)
rejected_first_pass = validated.where("exclusion_reason IS NOT NULL")
candidates = validated.where("exclusion_reason IS NULL").withColumn("station_id", F.trim("station_id"))
retained, rejected_duplicates = resolve_duplicate_keys(candidates)
clean = select_clean_columns(retained).persist(StorageLevel.DISK_ONLY)
rejected = rejected_first_pass.unionByName(rejected_duplicates, allowMissingColumns=True)
exclusions = rejected.groupBy("_source_year", "exclusion_reason").count()
quality = (
    raw.groupBy("_source_year").count().withColumnRenamed("count", "raw_rows")
    .join(clean.groupBy("year").agg(F.count("*").alias("retained_rows"), F.countDistinct("station_id").alias("stations")),
          F.col("_source_year") == F.col("year"), "left")
    .drop("year").withColumnRenamed("_source_year", "year").fillna(0)
)
clean.write.mode("overwrite").partitionBy("year").parquet(f"{ROOT}/cleaned")
rejected.write.mode("overwrite").partitionBy("_source_year").parquet(f"{ROOT}/rejected")
quality.write.mode("overwrite").json(f"{ROOT}/quality_by_year")
exclusions.write.mode("overwrite").json(f"{ROOT}/exclusion_counts")
alignment.write.mode("overwrite").json(f"{ROOT}/alignment_distribution")
quality.orderBy("year").show(8, truncate=False)
exclusions.orderBy("_source_year", "exclusion_reason").show(100, truncate=False)


## Coverage and missing intervals

Coverage uses the full calendar as its denominator, including absent station months. Gaps are measured between consecutive retained observations; no rainfall values are interpolated.

In [ ]:
months_observed = clean.groupBy("station_id", "year", "month").agg(
    F.count("*").alias("valid_slots"),
    F.countDistinct("local_date").alias("observed_days"),
    F.min("observation_time").alias("first_observation"),
    F.max("observation_time").alias("last_observation"),
)
calendar_years = spark.range(2017, 2025).select(F.col("id").cast("int").alias("year"))
calendar_months = spark.range(1, 13).select(F.col("id").cast("int").alias("month"))
station_calendar = clean.select("station_id").distinct().crossJoin(calendar_years).crossJoin(calendar_months)
months = (
    station_calendar.join(months_observed, ["station_id", "year", "month"], "left")
    .withColumn("expected_slots", F.dayofmonth(F.last_day(F.make_date("year", "month", F.lit(1)))) * 288)
    .fillna({"valid_slots": 0, "observed_days": 0})
    .withColumn("coverage", F.col("valid_slots") / F.col("expected_slots"))
)
years = months.groupBy("station_id", "year").agg(
    F.sum("valid_slots").alias("valid_slots"),
    F.sum("expected_slots").alias("expected_slots"),
).withColumn("coverage", F.col("valid_slots") / F.col("expected_slots"))
days = clean.groupBy("station_id", "local_date").agg(F.count("*").alias("valid_slots"))
days = days.withColumn("complete_day", F.col("valid_slots") == 288)
spacing = Window.partitionBy("station_id").orderBy("epoch")
gaps = clean.withColumn("elapsed_seconds", F.col("epoch") - F.lag("epoch").over(spacing)).where("elapsed_seconds > 300").groupBy("year", "elapsed_seconds").count()
for name, frame in [("coverage_month", months), ("coverage_year", years), ("coverage_day", days), ("gaps", gaps)]:
    frame.write.mode("overwrite").parquet(f"{ROOT}/{name}")
    frame.show(10, truncate=False)


In [ ]:
raw_count = sum(row["row_count"] for row in files)
excluded_count = rejected.count()
retained_count = clean.count()
display({"raw_rows": raw_count, "retained_rows": retained_count, "excluded_rows": excluded_count})
assert retained_count + excluded_count == raw_count


## Task 1 — rainfall patterns, 2017–2024

Task 1 describes all eight years. Monthly and hourly means average eligible station groups equally; they do not treat absent readings as zero rainfall. The 2024 descriptions are not used for Task 2 model or hyperparameter selection.


In [ ]:
task1_data = clean
train_wet_p99 = task1_data.where("year <= 2022 AND rain_mm > 0").agg(
    F.percentile_approx("rain_mm", 0.99, 10000).alias("p99")
).first()["p99"]

distribution_by_year = task1_data.groupBy("year").agg(
    F.count("*").alias("valid_intervals"),
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
    F.avg("rain_mm").alias("mean_mm_per_valid_interval"),
    F.avg(F.when(F.col("rain_mm") > 0, F.col("rain_mm"))).alias("conditional_wet_mean_mm"),
    F.percentile_approx(F.when(F.col("rain_mm") > 0, F.col("rain_mm")), [0.5, 0.9, 0.95, 0.99], 10000).alias("wet_quantiles_mm"),
    F.max("rain_mm").alias("max_observed_mm"),
).withColumn("training_wet_p99_mm", F.lit(float(train_wet_p99)))

station_month = task1_data.groupBy("station_id", "year", "month").agg(
    F.sum("rain_mm").alias("observed_total_mm"),
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
    F.avg(F.when(F.col("rain_mm") > 0, F.col("rain_mm"))).alias("wet_intensity_mm"),
    F.count("*").alias("valid_slots"),
).join(months.select("station_id", "year", "month", "coverage"), ["station_id", "year", "month"])
monthly = station_month.where(F.col("coverage") >= config["primary_coverage_cutoff"]).groupBy("year", "month").agg(
    F.avg("observed_total_mm").alias("mean_observed_total_mm"),
    F.avg("wet_frequency").alias("mean_wet_frequency"),
    F.avg("wet_intensity_mm").alias("mean_wet_intensity_mm"),
    F.count("*").alias("stations"),
)
distribution_by_year.orderBy("year").show(10, truncate=False)
monthly.orderBy("year", "month").show(84, truncate=False)


In [ ]:
station_annual = task1_data.groupBy("station_id", "year").agg(
    F.sum("rain_mm").alias("observed_total_mm"),
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
).join(years.where("year <= 2024"), ["station_id", "year"])
annual_results = []
for cutoff in config["coverage_cutoffs"]:
    common = years.where("year <= 2024").groupBy("station_id").agg(
        F.min("coverage").alias("minimum_coverage"), F.countDistinct("year").alias("years")
    ).where((F.col("minimum_coverage") >= cutoff) & (F.col("years") == 8))
    annual_results.append(
        station_annual.join(common.select("station_id"), "station_id").groupBy("year").agg(
            F.avg("observed_total_mm").alias("mean_observed_total_mm"),
            F.avg("wet_frequency").alias("mean_wet_frequency"),
            F.count("*").alias("common_stations"),
        ).withColumn("coverage_cutoff", F.lit(cutoff))
    )
annual_common_cohort = reduce(lambda left, right: left.unionByName(right), annual_results)


In [ ]:
station_hour = task1_data.groupBy("station_id", "year", "month", "hour").agg(
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
    F.count("*").alias("valid_slots"),
).join(months.select("station_id", "year", "month", "coverage", "expected_slots"), ["station_id", "year", "month"])
station_hour = station_hour.where(F.col("coverage") >= config["primary_coverage_cutoff"]).withColumn(
    "hour_coverage", F.col("valid_slots") / (F.col("expected_slots") / 24)
).where(F.col("hour_coverage") >= config["primary_coverage_cutoff"])
hourly_by_year = station_hour.groupBy("year", "hour").agg(
    F.avg("wet_frequency").alias("equal_station_month_wet_frequency"),
    F.count("*").alias("station_month_groups"),
)
annual_common_cohort.orderBy("coverage_cutoff", "year").show(30, truncate=False)
hourly_by_year.orderBy("year", "hour").show(200, truncate=False)


In [ ]:
daily = task1_data.groupBy("station_id", "year", "month", "local_date").agg(
    F.count("*").alias("valid_slots"), F.sum("rain_mm").alias("observed_daily_mm")
).withColumn("complete_day", F.col("valid_slots") == 288)
station_daily = daily.where("complete_day").groupBy("station_id", "year", "month").agg(
    F.avg("observed_daily_mm").alias("mean_complete_day_mm"),
    F.count("*").alias("complete_days"),
)
complete_daily = station_daily.groupBy("year", "month").agg(
    F.avg("mean_complete_day_mm").alias("equal_station_mean_complete_day_mm"),
    F.count("*").alias("stations"),
    F.sum("complete_days").alias("complete_station_days"),
)
calendar_days = months.where("year <= 2024").groupBy("year", "month").agg(
    F.sum(F.col("expected_slots") / 288).alias("fixed_network_calendar_days")
)
completeness = daily.groupBy("year", "month").agg(
    F.sum(F.col("complete_day").cast("long")).alias("complete_station_days")
).join(calendar_days, ["year", "month"], "right").fillna(0, ["complete_station_days"]).withColumn(
    "complete_fraction_of_calendar_days", F.col("complete_station_days") / F.col("fixed_network_calendar_days")
)

aggregates = {
    "distribution_by_year": distribution_by_year,
    "monthly": monthly,
    "annual_common_cohort": annual_common_cohort,
    "hourly_by_year": hourly_by_year,
    "complete_daily_patterns": complete_daily,
    "daily_completeness": completeness,
}
for name, frame in aggregates.items():
    frame.write.mode("overwrite").json(f"{ROOT}/task1_full/{name}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

figure_dir = PROJECT_ROOT / "report" / "figures" / "task1_full_period"
figure_dir.mkdir(parents=True, exist_ok=True)


def show_figure(fig, name):
    fig.tight_layout()
    fig.savefig(figure_dir / f"{name}.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)


In [ ]:
monthly_pd = monthly.orderBy("year", "month").toPandas()
hourly_pd = hourly_by_year.orderBy("year", "hour").toPandas()
annual_pd = annual_common_cohort.orderBy("coverage_cutoff", "year").toPandas()
distribution_pd = distribution_by_year.orderBy("year").toPandas()
coverage_pd = years.where("year <= 2024").select("station_id", "year", "coverage").toPandas()
complete_daily_pd = complete_daily.orderBy("year", "month").toPandas()
completeness_pd = completeness.orderBy("year", "month").toPandas()


## Read the results carefully

Wet frequency is calculated over valid five-minute observations. Monthly and hourly summaries are averages over eligible station groups. Annual comparisons require the same station in all eight analysed years at the selected coverage cutoff; if that cohort is empty, report the support limitation. Complete-day summaries first average days within each station, then average those station means equally; only days with exactly 288 readings contribute.


## Task 1 figures — generated from the analysis

The following code cells render figures from the Spark summary frames above and save the same figures under `report/figures/task1_full_period/`.


In [ ]:
coverage_matrix = coverage_pd.pivot(index="station_id", columns="year", values="coverage").sort_index()
fig, ax = plt.subplots(figsize=(9, 12))
image = ax.imshow(coverage_matrix.to_numpy(), aspect="auto", vmin=0, vmax=1, cmap="YlGnBu")
ax.set_xticks(range(len(coverage_matrix.columns)))
ax.set_xticklabels(coverage_matrix.columns)
ax.set_yticks(range(len(coverage_matrix.index)))
ax.set_yticklabels(coverage_matrix.index, fontsize=5)
ax.set(xlabel="Year", ylabel="Station", title="Annual observation coverage by station")
fig.colorbar(image, ax=ax, label="Valid slots / expected slots")
show_figure(fig, "coverage_heatmap")


In [ ]:
bin_width_mm = 0.1
overflow_bin = 200
wet_histogram_pd = (
    task1_data.where(F.col("rain_mm") > 0)
    .select(
        "year",
        F.when(F.col("rain_mm") >= 20.0, F.lit(overflow_bin))
        .otherwise(F.floor(F.col("rain_mm") / bin_width_mm).cast("int"))
        .alias("rain_bin"),
    )
    .groupBy("year", "rain_bin")
    .count()
    .orderBy("year", "rain_bin")
    .toPandas()
)


In [ ]:
wet_histogram = wet_histogram_pd.pivot(index="rain_bin", columns="year", values="count").fillna(0)
wet_bin_midpoints = [20.0 if value == overflow_bin else value * bin_width_mm + bin_width_mm / 2 for value in wet_histogram.index]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(distribution_pd["year"], distribution_pd["wet_frequency"] * 100, marker="o")
axes[0].set(xlabel="Year", ylabel="Wet readings (%)", title="Wet-record frequency")
for year in wet_histogram.columns:
    proportions = wet_histogram[year] / wet_histogram[year].sum()
    axes[1].step(wet_bin_midpoints, proportions, where="mid", label=str(year), alpha=0.8)
axes[1].axvline(float(train_wet_p99), color="black", linestyle="--", label="Training wet P99")
axes[1].set_yscale("log")
axes[1].set(xlabel="Rainfall per 5-minute reading (mm; final bin is ≥20 mm)", ylabel="Share of wet readings (log scale)", title="Conditional wet-reading distribution")
axes[1].legend(fontsize=7, ncol=2)
show_figure(fig, "rainfall_distribution")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for year, group in monthly_pd.groupby("year"):
    axes[0].plot(group["month"], group["mean_wet_frequency"] * 100, marker=".", label=str(year))
    axes[1].plot(group["month"], group["mean_wet_intensity_mm"], marker=".", label=str(year))
axes[0].set(xlabel="Month", ylabel="Wet readings (%)", xticks=range(1, 13), title="Wet frequency")
axes[1].set(xlabel="Month", ylabel="Mean wet reading (mm)", xticks=range(1, 13), title="Conditional wet intensity")
axes[1].legend(fontsize=7, ncol=2)
show_figure(fig, "monthly_frequency_intensity")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for year, group in monthly_pd.groupby("year"):
    axes[0].plot(group["month"], group["mean_observed_total_mm"], marker=".", label=str(year))
for year, group in hourly_pd.groupby("year"):
    axes[1].plot(group["hour"], group["equal_station_month_wet_frequency"], label=str(year))
axes[0].set(xlabel="Month", ylabel="Mean observed station-month rainfall (mm)", xticks=range(1, 13), title="Monthly rainfall")
axes[1].set(xlabel="Hour of interval end (SGT)", ylabel="Equal station-month wet frequency", xticks=range(0, 24, 3), title="Hourly wet frequency")
axes[0].legend(fontsize=7, ncol=2)
axes[1].legend(fontsize=7, ncol=2)
show_figure(fig, "monthly_hourly_patterns")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cutoff, group in annual_pd.groupby("coverage_cutoff"):
    label = f"Coverage ≥ {cutoff:.0%}"
    axes[0].plot(group["year"], group["mean_observed_total_mm"], marker="o", label=label)
    axes[1].plot(group["year"], group["mean_wet_frequency"] * 100, marker="o", label=label)
axes[0].set(xlabel="Year", ylabel="Mean observed annual station rainfall (mm)", title="Annual rainfall")
axes[1].set(xlabel="Year", ylabel="Mean wet frequency (%)", title="Annual wet frequency")
axes[1].legend(fontsize=8)
show_figure(fig, "annual_patterns")


## Additional Task 1 figures

These supplementary plots are also generated from the notebook's Spark summaries and exact five-minute observations.


In [ ]:
hourly_matrix = hourly_pd.pivot(index="year", columns="hour", values="equal_station_month_wet_frequency").sort_index()
fig, ax = plt.subplots(figsize=(11, 4))
image = ax.imshow(np.ma.masked_invalid(hourly_matrix.to_numpy()), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(hourly_matrix.columns)))
ax.set_xticklabels(hourly_matrix.columns)
ax.set_yticks(range(len(hourly_matrix.index)))
ax.set_yticklabels(hourly_matrix.index)
ax.set(xlabel="Hour of interval end (SGT)", ylabel="Year", title="Equal station-month wet frequency by year and hour")
fig.colorbar(image, ax=ax, label="Wet frequency")
show_figure(fig, "hourly_by_year")


In [ ]:
daily_matrix = completeness_pd.pivot(index="year", columns="month", values="complete_fraction_of_calendar_days").sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for year, group in complete_daily_pd.groupby("year"):
    axes[0].plot(group["month"], group["equal_station_mean_complete_day_mm"], marker=".", label=str(year))
image = axes[1].imshow(np.ma.masked_invalid(daily_matrix.to_numpy()), aspect="auto", vmin=0, vmax=1, cmap="YlGnBu")
axes[0].set(xlabel="Month", ylabel="Mean rainfall on complete station-days (mm)", xticks=range(1, 13))
axes[0].legend(fontsize=7, ncol=2)
axes[1].set_xticks(range(len(daily_matrix.columns)))
axes[1].set_xticklabels(daily_matrix.columns)
axes[1].set_yticks(range(len(daily_matrix.index)))
axes[1].set_yticklabels(daily_matrix.index)
axes[1].set(xlabel="Month", ylabel="Year", title="Complete station-days / calendar station-days")
fig.colorbar(image, ax=axes[1], label="Complete-day fraction")
show_figure(fig, "complete_daily_patterns")


In [ ]:
tail_exceedance_pd = (
    task1_data.groupBy("year")
    .agg(F.avg((F.col("rain_mm") > F.lit(float(train_wet_p99))).cast("double")).alias("exceedance_frequency"))
    .orderBy("year")
    .toPandas()
)
wet_quantiles = pd.DataFrame(distribution_pd["wet_quantiles_mm"].tolist(), columns=["P50", "P90", "P95", "P99"])
wet_quantiles.insert(0, "year", distribution_pd["year"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for quantile in ["P90", "P95", "P99"]:
    axes[0].plot(wet_quantiles["year"], wet_quantiles[quantile], marker="o", label=quantile)
axes[0].axhline(float(train_wet_p99), color="black", linestyle="--", label="Training wet P99")
axes[0].set(xlabel="Year", ylabel="Wet-reading rainfall (mm)", title="Upper wet-reading quantiles")
axes[0].legend()
axes[1].plot(tail_exceedance_pd["year"], tail_exceedance_pd["exceedance_frequency"] * 100, marker="o")
axes[1].set(xlabel="Year", ylabel="Valid readings above training wet P99 (%)", title=f"Exceedance of {train_wet_p99:.1f} mm")
show_figure(fig, "rainfall_tail")


In [ ]:
sequence_window = Window.partitionBy("station_id").orderBy("epoch")
run_window = sequence_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
spell_rows = (
    task1_data.select("station_id", "year", "epoch", (F.col("rain_mm") > 0).cast("int").alias("is_wet"))
    .withColumn("previous_epoch", F.lag("epoch").over(sequence_window))
    .withColumn("previous_wet", F.lag("is_wet").over(sequence_window))
    .withColumn("next_epoch", F.lead("epoch").over(sequence_window))
    .withColumn("left_censored", F.col("previous_epoch").isNull() | ((F.col("epoch") - F.col("previous_epoch")) != 300))
    .withColumn("right_censored", F.col("next_epoch").isNull() | ((F.col("next_epoch") - F.col("epoch")) != 300))
    .withColumn("new_run", F.col("left_censored") | (F.col("is_wet") != F.col("previous_wet")))
    .withColumn("run_id", F.sum(F.col("new_run").cast("int")).over(run_window))
)


In [ ]:
wet_runs = (
    spell_rows.where("is_wet = 1")
    .groupBy("station_id", "run_id")
    .agg(
        F.min("year").alias("start_year"),
        F.count("*").alias("readings"),
        F.max(F.col("left_censored").cast("int")).alias("left_censored"),
        F.max(F.col("right_censored").cast("int")).alias("right_censored"),
    )
)
wet_spells_pd = (
    wet_runs.groupBy("start_year")
    .agg(
        F.percentile_approx("readings", [0.5, 0.9, 0.99], 10000).alias("run_quantiles"),
        F.count("*").alias("spell_count"),
        F.avg("left_censored").alias("left_censored_fraction"),
        F.avg("right_censored").alias("right_censored_fraction"),
    )
    .orderBy("start_year")
    .toPandas()
)


In [ ]:
spell_quantiles = pd.DataFrame(wet_spells_pd["run_quantiles"].tolist(), columns=["P50", "P90", "P99"])
spell_quantiles.insert(0, "year", wet_spells_pd["start_year"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for quantile in ["P50", "P90", "P99"]:
    axes[0].plot(spell_quantiles["year"], spell_quantiles[quantile], marker="o", label=quantile)
axes[0].set_yscale("log")
axes[0].set(xlabel="Start year", ylabel="Consecutive wet readings (log scale)", title="Exact 5-minute wet spells")
axes[0].legend()
axes[1].plot(wet_spells_pd["start_year"], wet_spells_pd["left_censored_fraction"] * 100, marker="o", label="Gap/start boundary")
axes[1].plot(wet_spells_pd["start_year"], wet_spells_pd["right_censored_fraction"] * 100, marker="o", label="Gap/end boundary")
axes[1].set(xlabel="Start year", ylabel="Censored wet spells (%)", title="Spells censored by missing intervals or study bounds")
axes[1].legend()
show_figure(fig, "wet_spell_patterns")


In [ ]:
spatial_cutoff = min(config["coverage_cutoffs"])
stable_locations = (
    task1_data.groupBy("station_id")
    .agg(
        F.min("longitude").alias("min_longitude"), F.max("longitude").alias("max_longitude"),
        F.min("latitude").alias("min_latitude"), F.max("latitude").alias("max_latitude"),
    )
    .where(((F.col("max_longitude") - F.col("min_longitude")) <= 1e-6) & ((F.col("max_latitude") - F.col("min_latitude")) <= 1e-6))
    .select("station_id", F.col("min_longitude").alias("longitude"), F.col("min_latitude").alias("latitude"))
)
spatial_pd = (
    station_annual.join(years.select("station_id", "year", "coverage"), ["station_id", "year"])
    .where((F.col("year") <= 2023) & (F.col("coverage") >= spatial_cutoff))
    .join(stable_locations, "station_id")
    .select("station_id", "year", "longitude", "latitude", "observed_total_mm")
    .orderBy("year", "station_id")
    .toPandas()
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
points = ax.scatter(spatial_pd["longitude"], spatial_pd["latitude"], c=spatial_pd["observed_total_mm"], s=28, cmap="viridis", alpha=0.8, edgecolors="white", linewidths=0.3)
ax.set(xlabel="Longitude", ylabel="Latitude", title=f"Station-years with ≥{spatial_cutoff:.0%} annual coverage")
ax.grid(alpha=0.2)
fig.colorbar(points, ax=ax, label="Observed annual rainfall (mm)")
show_figure(fig, "station_spatial_patterns")


In [ ]:
raw.unpersist()
clean.unpersist()
print("Cleaning and full-period Task 1 summaries are saved under the selected run prefix.")
